# NB_hospital_daily_summary

Builds `<catalog>.gold.hospital_daily_summary` (one row per hospital per day with lab activity) from silver lab_results, joined to hospitals and the calendar dimension, with a LAG-based day-over-day trend, per the STTM `Gold/sttm_silver_to_gold.xlsx`. Load type: FULL rebuild (CREATE OR REPLACE TABLE ... AS SELECT).

In [0]:
# Catalog placeholder from the STTM (<catalog>); can be overridden by ADF / Jobs
dbutils.widgets.text("catalog", "adb_caresync")
catalog = dbutils.widgets.get("catalog").strip() or "adb_caresync"

silver = f"{catalog}.silver"
target_table = f"{catalog}.gold.hospital_daily_summary"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")
print(f"Source schema : {silver}")
print(f"Target table  : {target_table}")

In [0]:
# Driving table: silver.lab_results grouped by (hospital_id, test_date)
# LEFT JOIN hospitals (hospital_name) and calendar (day_name, is_weekend) on date_key = yyyyMMdd(test_date)
# LAG(total_tests) per hospital ordered by summary_date -> previous day WITH lab activity
sql_text = f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per hospital per day with lab-test volumes, abnormal/critical rates and day-over-day trend (source: STTM silver_to_gold)'
AS
WITH hospitals AS (
  SELECT hospital_id, hospital_name
  FROM {silver}.hospitals
  QUALIFY ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY update_timestamp DESC) = 1
),
daily AS (
  SELECT
    hospital_id,
    test_date                                    AS summary_date,
    COUNT(*)                                     AS total_tests,
    COUNT(DISTINCT patient_id)                   AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                    AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')         AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')         AS critical_test_count
  FROM {silver}.lab_results
  GROUP BY hospital_id, test_date
),
trend AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(t.hospital_id AS INT)                                                     AS hospital_id,
  h.hospital_name                                                                AS hospital_name,
  CAST(t.summary_date AS DATE)                                                   AS summary_date,
  c.day_name                                                                     AS day_name,
  c.is_weekend                                                                   AS is_weekend,
  CAST(t.total_tests AS INT)                                                     AS total_tests,
  CAST(t.unique_patients_tested AS INT)                                          AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                                             AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                                             AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                             AS critical_test_count,
  CAST(ROUND(t.abnormal_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2))  AS abnormal_rate_pct,
  CAST(ROUND(t.critical_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2))  AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                                            AS prev_day_total_tests,
  CASE
    WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
    ELSE CAST(ROUND((t.total_tests - t.prev_day_total_tests) * 100.0 / t.prev_day_total_tests, 2) AS DECIMAL(6,2))
  END                                                                            AS test_volume_change_pct
FROM trend t
LEFT JOIN hospitals h               ON t.hospital_id = h.hospital_id
LEFT JOIN {silver}.calendar c       ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT)
"""
print(sql_text)
spark.sql(sql_text)
print(f"Created {target_table}")

In [0]:
validation = spark.sql(f"""
  SELECT
    COUNT(*)                                                AS total_rows,
    COUNT(DISTINCT hospital_id, summary_date)               AS distinct_hospital_days,
    SUM(total_tests)                                        AS total_tests,
    (SELECT COUNT(*) FROM {silver}.lab_results)             AS source_lab_results,
    COUNT_IF(hospital_name IS NULL)                         AS missing_hospital_name,
    COUNT_IF(day_name IS NULL)                              AS missing_calendar_match,
    MIN(summary_date)                                       AS min_date,
    MAX(summary_date)                                       AS max_date
  FROM {target_table}
""")
display(validation)

r = validation.first()
assert r.total_rows == r.distinct_hospital_days, "Grain violated: expected one row per hospital per day"
assert r.total_tests == r.source_lab_results, "Test totals do not reconcile with silver.lab_results"
print("Validation passed: one row per hospital per day, totals reconcile.")

In [0]:
# Return the gold record count to the caller (ADF / Jobs / dbutils.notebook.run)
updated_df = spark.table(target_table)
record_count = updated_df.count()
print(f"{target_table} record count: {record_count}")
dbutils.notebook.exit(str(record_count))